# ⚡ Fabric Spark Pool Optimiser

Analyses real Spark session data across your Fabric workspaces and recommends the optimal Spark pool configuration.

---

### How it works
1. **Auto-discovers** all workspaces you have access to
2. **Detects Workspace Monitoring** — uses real vCores where available
3. **Classifies sessions** — separates automated (pipeline/SPN) from interactive (dev) sessions
4. **Collects Spark sessions** via the Fabric Monitoring API (1 call per workspace)
5. **Analyses** duration, data movement (GB via stages), and parallelism
6. **Recommends** optimal nodeSize + maxNodes with full explanation and config steps
7. **Estimates** monthly CU savings vs current pool
8. **Renders an interactive dashboard** with an About page explaining the methodology

### Requirements
- **Spark Notebook** in Microsoft Fabric · Viewer role on workspaces · No lakehouse needed

### Data sources & known limitations
| Data | Source | Notes |
|------|--------|-------|
| Session duration | Livy Sessions API | Real |
| GB read/written/shuffle | Spark History Stages | Real — not attributable to sub-notebooks |
| Max concurrent sessions | Livy Sessions API | Real |
| vCores really used | Workspace Monitoring KQL | Only if monitoring enabled |
| vCores really used | resourceusage API | ❌ 500 for pipeline-triggered sessions |
| Sub-notebook breakdown | itemsnapshots / jobGroups | ❌ Not available (all jobGroup=None) |


In [ ]:
# ============================================================
# CELL 1: CONFIGURATION
# ============================================================

import sempy.fabric as fabric
import pandas as pd
import numpy as np
import json
import warnings
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ── WORKSPACE FILTER ─────────────────────────────────────────
# Leave empty [] to analyse ALL workspaces.
# Or specify names: ['PRO_WECODATA_00_CONTROL', 'PRO_WECODATA_02_SILVER']
WORKSPACE_FILTER = []

# ── ANALYSIS SETTINGS ────────────────────────────────────────
DAYS_BACK              = 7     # Days of history
MIN_RUNS               = 3     # Min sessions for a recommendation
MAX_SESSIONS_PER_WS    = 50    # Cap per workspace
MARGIN_PCT             = 1.20  # 20% headroom above p95
PARALLEL_WORKERS       = 4     # Threads for enrichment
INTERACTIVE_THRESHOLD  = 1800  # Sessions > 30min by a user = interactive dev

# ── ORCHESTRATOR DETECTION ───────────────────────────────────
ORCHESTRATOR_PATTERNS = ['Orch', 'Orchestrat', 'Pipeline', 'Daily_', 'PL_']

client = fabric.FabricRestClient()

print('✅ Configuration ready')
print(f'   Period    : last {DAYS_BACK} days')
print(f'   Filter    : {WORKSPACE_FILTER if WORKSPACE_FILTER else "all workspaces"}')
print(f'   Min runs  : {MIN_RUNS}')
print(f'   Dev threshold: >{INTERACTIVE_THRESHOLD}s by user = interactive session')

In [ ]:
# ============================================================
# CELL 2: API + HELPER FUNCTIONS
# ============================================================

def fix_types(obj):
    """Convert numpy/pandas types to Python native for JSON serialization."""
    if isinstance(obj, dict):
        return {k: fix_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [fix_types(i) for i in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    elif obj != obj:
        return None
    return obj


def classify_session(row):
    """
    Classify a Spark session as 'automated' or 'interactive'.

    Automated: Service Principal submitter, orchestrator pattern, short duration.
    Interactive: User submitter + long duration (developer leaving notebook open).

    Interactive sessions skew avg_duration → excluded from CU/timing analysis.
    Their stages data (GB) is still real and used for nodeSize sizing.
    """
    submitter_type = row.get('submitter_type', 'unknown')
    duration_sec   = row.get('duration_sec', 0) or 0
    is_orch        = row.get('is_orchestrator', False)

    if submitter_type == 'ServicePrincipal':
        return 'automated'
    if is_orch:
        return 'automated'
    if duration_sec > INTERACTIVE_THRESHOLD and submitter_type in ['Member', 'User', 'unknown']:
        return 'interactive'
    return 'automated'


def get_all_workspaces():
    try:
        resp = client.get('v1/workspaces')
        return [{'name': w['displayName'], 'id': w['id']}
                for w in resp.json().get('value', [])]
    except Exception as e:
        print(f'  ⚠ Could not list workspaces: {e}')
        return []


def get_all_sessions_for_workspace(workspace_id):
    """Single paginated call for ALL Spark sessions in a workspace."""
    since = (datetime.utcnow() - timedelta(days=DAYS_BACK)).strftime('%Y-%m-%dT%H:%M:%SZ')
    sessions, page = [], 0
    url = f'v1/workspaces/{workspace_id}/spark/livySessions?startDateTime={since}'
    while url and page < 20:
        try:
            data = client.get(url).json()
            sessions.extend(data.get('value', []))
            page += 1
            nu = data.get('continuationUri')
            nt = data.get('continuationToken')
            if nu:   url = nu
            elif nt: url = f'v1/workspaces/{workspace_id}/spark/livySessions?startDateTime={since}&continuationToken={nt}'
            else:    url = None
        except: break
    if not sessions:
        return pd.DataFrame()
    rows = []
    for s in sessions:
        nb_name = s.get('itemName', 'unknown')
        is_orch = any(p.lower() in nb_name.lower() for p in ORCHESTRATOR_PATTERNS)
        rows.append({
            'livy_id':        s.get('livyId', ''),
            'notebook_id':    s.get('item', {}).get('itemId', ''),
            'notebook_name':  nb_name,
            'submitted_by':   s.get('submitter', {}).get('id', 'unknown'),
            'submitter_type': s.get('submitter', {}).get('type', 'unknown'),
            'start_time':     s.get('startDateTime'),
            'end_time':       s.get('endDateTime'),
            'duration_sec':   s.get('runningDuration', {}).get('value', 0),
            'status':         s.get('state', 'unknown'),
            'app_id':         s.get('sparkApplicationId', ''),
            'item_type':      s.get('itemType', 'Notebook'),
            'is_orchestrator':is_orch,
        })
    return pd.DataFrame(rows)


def get_data_movement(workspace_id, notebook_id, livy_id, app_id):
    """
    Data movement from /stages endpoint.
    /jobs endpoint exists but does NOT contain inputBytes/outputBytes.
    For orchestrator sessions (runMultiple), all sub-notebook jobs appear
    under jobGroup=None — GB cannot be attributed to individual sub-notebooks.
    """
    try:
        resp = client.get(
            f'v1/workspaces/{workspace_id}/notebooks/{notebook_id}'
            f'/livySessions/{livy_id}/applications/{app_id}/stages'
        )
        if resp.status_code != 200:
            return _edm()
        stages = resp.json()
        if not isinstance(stages, list) or not stages:
            return _edm()
        return {
            'input_gb':      round(sum(s.get('inputBytes',       0) for s in stages) / 1e9, 4),
            'output_gb':     round(sum(s.get('outputBytes',      0) for s in stages) / 1e9, 4),
            'shuffle_gb':    round(sum(s.get('shuffleReadBytes', 0) for s in stages) / 1e9, 4),
            'peak_mem_gb':   round(max((s.get('peakExecutionMemory', 0) for s in stages), default=0) / 1e9, 4),
            'input_records': sum(s.get('inputRecords',  0) for s in stages),
            'output_records':sum(s.get('outputRecords', 0) for s in stages),
        }
    except:
        return _edm()


def _edm():
    return {'input_gb':0,'output_gb':0,'shuffle_gb':0,
            'peak_mem_gb':0,'input_records':0,'output_records':0}


def enrich_session(ws_id, row):
    dat = get_data_movement(ws_id, row['notebook_id'], row['livy_id'], row['app_id'])
    return {**row, **dat}


def get_pool_config(workspace_id):
    try:
        pools = client.get(f'v1/workspaces/{workspace_id}/spark/pools').json().get('value', [])
        if pools:
            p = pools[0]
            return {
                'cur_node_size': p.get('nodeSize', 'Medium'),
                'cur_max_nodes': p.get('maxNodeCount', 4),
                'cur_min_nodes': p.get('minNodeCount', 1),
                'cur_autoscale': p.get('autoScaleEnabled', True),
                'pool_name':     p.get('name', 'StarterPool'),
            }
    except: pass
    return {'cur_node_size':'Medium','cur_max_nodes':4,
            'cur_min_nodes':1,'cur_autoscale':True,'pool_name':'StarterPool'}


def get_notebook_count(workspace_id):
    try:
        return len(client.get(f'v1/workspaces/{workspace_id}/notebooks').json().get('value', []))
    except: return 0


def calc_max_parallel(df):
    """Max concurrent Spark sessions observed."""
    df = df.copy()
    df['start_dt'] = pd.to_datetime(df['start_time'], errors='coerce')
    df['end_dt']   = pd.to_datetime(df['end_time'],   errors='coerce')
    df = df.dropna(subset=['start_dt', 'end_dt'])
    if df.empty: return 1
    return max(
        len(df[(df['start_dt'] <= r['end_dt']) & (df['end_dt'] >= r['start_dt'])])
        for _, r in df.iterrows()
    )


def detect_workspace_monitoring(workspaces):
    """Auto-detect Workspace Monitoring KQL databases."""
    monitoring_map = {}
    for ws in workspaces:
        try:
            resp = client.get(f'v1/workspaces/{ws["id"]}/kqlDatabases')
            if resp.status_code != 200: continue
            for db in resp.json().get('value', []):
                if 'monitor' in db.get('displayName', '').lower():
                    monitoring_map[ws['id']] = {
                        'ws_name': ws['name'],
                        'db_id':   db['id'],
                        'db_name': db['displayName'],
                    }
                    print(f'  ✅ Monitoring: {ws["name"]} → {db["displayName"]}')
        except: continue
    return monitoring_map


def get_vcores_from_monitoring(workspace_id, kql_db_id):
    """Query Workspace Monitoring KQL for real vCore usage."""
    since = (datetime.utcnow() - timedelta(days=DAYS_BACK)).strftime('%Y-%m-%dT%H:%M:%SZ')
    kql = f"""
    SparkApplications
    | where StartTime >= datetime({since})
    | summarize
        vcores_p95   = percentile(ExecutorCores, 95),
        vcores_max   = max(ExecutorCores),
        idle_pct_avg = iff(avg(AllocatedCores)>0, 1.0-avg(ExecutorCores)/avg(AllocatedCores), 1.0)
      by ApplicationId, ItemName
    """
    try:
        resp = client.post(
            f'v1/workspaces/{workspace_id}/kqlDatabases/{kql_db_id}/query',
            json={'query': kql}
        )
        if resp.status_code != 200: return pd.DataFrame()
        tbl  = resp.json().get('tables', [{}])[0]
        rows = tbl.get('rows', [])
        cols = [c['name'] for c in tbl.get('columns', [])]
        return pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame()
    except: return pd.DataFrame()


print('✅ API + helper functions ready')

In [ ]:
# ============================================================
# CELL 3: RECOMMENDATION ENGINE
# ============================================================

# 1 CU = 2 Spark vCores (Microsoft official docs)
NODE_SPECS = {
    'Small':   {'vcores': 4,  'ram_gb': 32,  'cu_hr': 2},
    'Medium':  {'vcores': 8,  'ram_gb': 64,  'cu_hr': 4},
    'Large':   {'vcores': 16, 'ram_gb': 128, 'cu_hr': 8},
    'XLarge':  {'vcores': 32, 'ram_gb': 256, 'cu_hr': 16},
    'XXLarge': {'vcores': 64, 'ram_gb': 512, 'cu_hr': 32},
}
SIZE_ORDER = ['Small', 'Medium', 'Large', 'XLarge', 'XXLarge']


def recommend_pool(stats):
    """
    Three-tier recommendation:

    TIER 1 — Workspace Monitoring vCores (HIGH confidence):
      nodeSize = smallest node where vCores >= p95_real x 1.20

    TIER 2 — Stage data movement GB/min (MED confidence):
      nodeSize inferred from throughput:
        > 5 GB/min → Large · > 1 GB/min → Medium · else → Small
      Used also when mostly_interactive (duration unreliable, GB still real)

    TIER 3 — Duration + parallelism only (LOW confidence):
      nodeSize = keep current (cannot determine if over/under sized)
      maxNodes = from observed parallelism

    CU formula (Microsoft official):
      CU = cu_per_node_hr x max_nodes x avg_duration_hr x runs_per_month
      saving = cu_current - cu_recommended
      positive = saving (downsize), negative = extra cost (upsize)

    Interactive dev session handling:
      If >50% sessions are interactive (user + duration > threshold):
        - Duration excluded from CU calculation (unreliable)
        - GB-based sizing forced (Tier 2)
        - Automated session duration used if available
    """
    vcores_p95         = stats.get('vcores_p95')
    vcores_source      = stats.get('vcores_source', 'none')
    cur_size           = stats.get('cur_node_size', 'Medium')
    cur_max            = stats.get('cur_max_nodes', 4)
    max_par            = stats.get('max_parallel', 1)
    avg_input_gb       = stats.get('avg_input_gb', 0)
    avg_shuffle_gb     = stats.get('avg_shuffle_gb', 0)
    n_runs             = stats.get('sample_size', 0)
    is_orch            = stats.get('is_orchestrator_ws', False)
    mostly_interactive = stats.get('mostly_interactive', False)
    # Use automated session duration for CU (interactive excluded)
    avg_dur_sec        = stats.get('avg_duration_automated_sec') or stats.get('avg_duration_sec', 300)
    n_automated        = stats.get('automated_sessions', n_runs)

    memory_note = ''

    # ── TIER 1: Real vCores from Workspace Monitoring ─────────
    if vcores_p95 and vcores_p95 > 0 and vcores_source == 'monitoring' and not mostly_interactive:
        needed    = vcores_p95 * MARGIN_PCT
        node_size = SIZE_ORDER[-1]
        for s in SIZE_ORDER:
            if NODE_SPECS[s]['vcores'] >= needed:
                node_size = s
                break
        sizing_basis    = f'Real vCores p95: {vcores_p95:.1f} → needs {needed:.1f} with {int((MARGIN_PCT-1)*100)}% margin → {node_size}'
        confidence_base = 'HIGH'

    # ── TIER 2: GB/min from stages ────────────────────────────
    elif avg_input_gb > 0 or avg_shuffle_gb > 0:
        total_gb   = avg_input_gb + avg_shuffle_gb
        dur_min    = max(avg_dur_sec / 60, 0.1)
        gb_per_min = total_gb / dur_min
        if   gb_per_min > 5:   node_size = 'Large'
        elif gb_per_min > 1:   node_size = 'Medium'
        else:                   node_size = 'Small'
        note = ' (duration from automated sessions only)' if mostly_interactive else ''
        sizing_basis    = f'Data throughput: {total_gb:.3f} GB / {dur_min:.1f} min = {gb_per_min:.2f} GB/min → {node_size}{note}'
        confidence_base = 'MED'

    # ── TIER 3: No data → keep current nodeSize ───────────────
    else:
        node_size       = cur_size
        sizing_basis    = 'No data movement detected. nodeSize unchanged — enable Workspace Monitoring for vCore-based sizing.'
        confidence_base = 'LOW'

    # ── Memory pressure check ─────────────────────────────────
    if avg_input_gb > 0 and avg_shuffle_gb > avg_input_gb * 1.5:
        idx = SIZE_ORDER.index(node_size)
        if idx < len(SIZE_ORDER) - 1:
            node_size    = SIZE_ORDER[idx + 1]
            memory_note  = f'Upsized one level: shuffle ({avg_shuffle_gb:.3f} GB) > 1.5x input ({avg_input_gb:.3f} GB)'

    # ── maxNodes from observed parallelism ────────────────────
    # ── maxNodes from observed parallelism ────────────────────────
    vc_per_node   = NODE_SPECS[node_size]['vcores']
    # 1 node per job minimum — don't multiply by nodes_per_job
    # (that was inflating maxNodes when downsizing nodeSize)
    rec_max = max(1, round(max_par * 1.25))

    # ── Action ────────────────────────────────────────────────────
    size_diff = SIZE_ORDER.index(node_size) - SIZE_ORDER.index(cur_size)
    if   size_diff < 0 or (size_diff == 0 and rec_max < cur_max): action = 'DOWNSIZE'
    elif size_diff > 0 or (size_diff == 0 and rec_max > cur_max): action = 'UPSIZE'
    else:                                                           action = 'OK'

    # ── CU saving ─────────────────────────────────────────────────
    runs_per_mo = max(n_automated / (DAYS_BACK / 30), 1)
    avg_dur_hr  = max(avg_dur_sec / 3600, 0.01)
    cu_cur      = NODE_SPECS[cur_size]['cu_hr']  * cur_max  * avg_dur_hr * runs_per_mo
    cu_rec      = NODE_SPECS[node_size]['cu_hr'] * rec_max  * avg_dur_hr * runs_per_mo
    cu_saving   = round(cu_cur - cu_rec, 1)

    # ── Coherence check: DOWNSIZE should not increase cost ────────
    if action == 'DOWNSIZE' and cu_saving < 0:
        # nodeSize going down is correct BUT maxNodes going up too much
        # → try keeping current maxNodes with the smaller nodeSize
        cu_rec_alt    = NODE_SPECS[node_size]['cu_hr'] * cur_max * avg_dur_hr * runs_per_mo
        cu_saving_alt = round(cu_cur - cu_rec_alt, 1)

        if cu_saving_alt > 0:
            # Best of both: smaller nodeSize + same maxNodes = real saving
            rec_max   = cur_max
            cu_rec    = cu_rec_alt
            cu_saving = cu_saving_alt
            sizing_basis += f' | maxNodes kept at {cur_max} (reducing from {round(max_par * 1.25)} avoids cost increase)'
        else:
            # Can't save with this nodeSize either → no change
            action    = 'OK'
            node_size = cur_size
            rec_max   = cur_max
            cu_rec    = cu_cur
            cu_saving = 0
            sizing_basis += ' | No cost-saving configuration found with available data'

    # ── Confidence ────────────────────────────────────────────
    if   confidence_base == 'HIGH' and n_runs >= MIN_RUNS * 4: confidence = 'HIGH'
    elif confidence_base == 'MED'  and n_runs >= MIN_RUNS:     confidence = 'MED'
    else:                                                        confidence = 'LOW'

    # ── Explanation ───────────────────────────────────────────
    interactive_note = ''
    if mostly_interactive:
        n_int = stats.get('interactive_sessions', 0)
        interactive_note = f'{n_int} interactive dev sessions detected (duration >{INTERACTIVE_THRESHOLD}s by user) — excluded from timing/CU analysis. GB-based sizing used instead.'

    explanation = ' | '.join(filter(None, [
        f'Data source: {vcores_source.upper()}',
        interactive_note,
        sizing_basis,
        f'Max concurrent sessions observed: {max_par} → {rec_max} nodes (x1.25 buffer)',
        f'Avg automated session duration: {avg_dur_sec:.0f}s ({avg_dur_hr:.2f}h)',
        f'CU formula: {NODE_SPECS[node_size]["cu_hr"]} CU/hr x {rec_max} nodes x {avg_dur_hr:.2f}h x {runs_per_mo:.0f} runs/mo = {cu_rec:.0f} CU/mo',
        f'Current: {cu_cur:.0f} CU/mo → Recommended: {cu_rec:.0f} CU/mo → Net: {cu_saving:+.0f} CU/mo',
        f'⚠ Memory pressure: {memory_note}' if memory_note else '',
        'ℹ Orchestrator workspace: pool shared by all sub-notebooks via runMultiple()' if is_orch else '',
        'ℹ Enable Workspace Monitoring for vCore-based sizing (more accurate)' if vcores_source == 'none' else '',
    ]))

    config_steps = [
        'Open this workspace in Microsoft Fabric',
        'Go to Workspace settings (gear icon top right)',
        'Navigate to: Data Engineering / Science → Spark settings → Pool tab',
        'Edit the existing pool or click New pool',
        f'Node size: {node_size}  ({NODE_SPECS[node_size]["vcores"]} vCores · {NODE_SPECS[node_size]["ram_gb"]} GB RAM · {NODE_SPECS[node_size]["cu_hr"]} CU/hr per node)',
        'Min nodes: 1',
        f'Max nodes: {rec_max}',
        'Enable Autoscale: Yes',
        'Enable Dynamic executor allocation: Yes',
        'Save — takes effect on the next Spark session start',
    ]

    return {
        'rec_node_size':     node_size,
        'rec_max_nodes':     rec_max,
        'rec_min_nodes':     1,
        'action':            action,
        'memory_note':       memory_note,
        'cu_saving_mo':      cu_saving,
        'cu_current_mo':     round(cu_cur, 1),
        'cu_recommended_mo': round(cu_rec, 1),
        'explanation':       explanation,
        'config_steps':      config_steps,
        'confidence':        confidence,
        'vcores_source':     vcores_source,
    }


print('✅ Recommendation engine ready')
print('   Tier 1: Workspace Monitoring vCores → HIGH confidence')
print('   Tier 2: Stage GB/min (or forced if mostly interactive) → MED confidence')
print('   Tier 3: Duration + parallelism only → LOW confidence')
print('   Interactive dev sessions excluded from timing/CU analysis')

In [ ]:
# ============================================================
# CELL 4: COLLECT & ANALYSE
# ============================================================

print('🔍 Discovering workspaces...')
all_ws = get_all_workspaces()

if WORKSPACE_FILTER:
    workspaces = [w for w in all_ws if w['name'] in WORKSPACE_FILTER]
    print(f'   Filtered to {len(workspaces)} / {len(all_ws)} workspaces')
else:
    workspaces = all_ws
    print(f'   Found {len(workspaces)} workspaces')

print('\n🔍 Checking for Workspace Monitoring...')
MONITORING_MAP = detect_workspace_monitoring(workspaces)
if MONITORING_MAP:
    print(f'   ✅ Monitoring active in {len(MONITORING_MAP)} workspace(s) → vCore-based sizing')
else:
    print('   ⚠ No Workspace Monitoring found → estimation-based sizing')
    print('   → To enable: Admin Portal → Workspace Settings → Monitoring')

print()
all_results    = []
all_nb_details = {}

for ws in workspaces:
    ws_name = ws['name']
    ws_id   = ws['id']
    pool    = get_pool_config(ws_id)

    print(f'  ⟳  {ws_name}', end=' ... ')

    apps_df = get_all_sessions_for_workspace(ws_id)

    if apps_df.empty or len(apps_df) < MIN_RUNS:
        nb_count = get_notebook_count(ws_id)
        print(f'no sessions ({len(apps_df) if not apps_df.empty else 0} runs, {nb_count} notebooks)')
        all_results.append(fix_types({
            'workspace_name': ws_name, 'workspace_id': ws_id,
            'has_notebooks': nb_count > 0, 'nb_count': nb_count,
            'sample_size': len(apps_df) if not apps_df.empty else 0,
            'action': 'NO_DATA', 'rec_node_size': pool['cur_node_size'],
            'rec_max_nodes': pool['cur_max_nodes'], 'rec_min_nodes': 1,
            'cu_saving_mo': 0, 'cu_current_mo': 0, 'cu_recommended_mo': 0,
            'confidence': 'LOW', 'vcores_source': 'none',
            'vcores_p95': None, 'idle_pct_avg': None,
            'avg_input_gb': 0, 'avg_shuffle_gb': 0, 'avg_duration_sec': 0,
            'avg_duration_automated_sec': 0, 'max_parallel': 1,
            'automated_sessions': 0, 'interactive_sessions': 0,
            'mostly_interactive': False, 'is_orchestrator_ws': False,
            'top_notebook': 'N/A', 'top_user': 'N/A', 'memory_note': '',
            'explanation': 'No Spark sessions found in the selected period.',
            'config_steps': [], **pool,
        }))
        continue

    if len(apps_df) > MAX_SESSIONS_PER_WS:
        apps_df = apps_df.sort_values('start_time', ascending=False).head(MAX_SESSIONS_PER_WS)

    # ── Enrich ───────────────────────────────────────────────
    rows     = apps_df.to_dict('records')
    enriched = []
    with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
        futures = {ex.submit(enrich_session, ws_id, r): r for r in rows}
        for f in as_completed(futures):
            try:    enriched.append(f.result())
            except: enriched.append(futures[f])
    df = pd.DataFrame(enriched)

    # ── Classify sessions ────────────────────────────────────
    df['session_type']   = df.apply(classify_session, axis=1)
    automated_df         = df[df['session_type'] == 'automated']
    interactive_df       = df[df['session_type'] == 'interactive']
    n_automated          = int(len(automated_df))
    n_interactive        = int(len(interactive_df))
    mostly_interactive   = bool(n_interactive / len(df) > 0.5) if len(df) > 0 else False
    analysis_df          = automated_df if n_automated >= MIN_RUNS else df

    # ── Orchestrator detection ───────────────────────────────
    orch_ratio  = float(df['is_orchestrator'].mean()) if 'is_orchestrator' in df else 0
    spn_ratio   = float((df['submitter_type'] == 'ServicePrincipal').mean())
    is_orch_ws  = bool(orch_ratio >= 0.3 or spn_ratio >= 0.5)

    # ── Workspace Monitoring vCores ──────────────────────────
    vcores_p95_final = None
    idle_pct_final   = None
    vcores_source    = 'none'

    if ws_id in MONITORING_MAP:
        mon    = MONITORING_MAP[ws_id]
        mon_df = get_vcores_from_monitoring(ws_id, mon['db_id'])
        if not mon_df.empty and 'vcores_p95' in mon_df.columns:
            vcores_p95_final = round(float(mon_df['vcores_p95'].quantile(0.95)), 1)
            idle_pct_final   = round(float(mon_df['idle_pct_avg'].mean()), 2)
            vcores_source    = 'monitoring'
            print(f'[MON ✅ vCores p95={vcores_p95_final}]', end=' ')

    if vcores_source == 'none':
        vcores_source = 'stages' if float(df.get('input_gb', pd.Series([0])).sum()) > 0 else 'none'

    # ── Notebook breakdown ───────────────────────────────────
    nb_details = []
    for nb_name, nb_df in df.groupby('notebook_name'):
        nb_auto = nb_df[nb_df['session_type'] == 'automated']
        nb_details.append(fix_types({
            'notebook_name':    str(nb_name),
            'runs':             int(len(nb_df)),
            'automated_runs':   int(len(nb_auto)),
            'interactive_runs': int(len(nb_df) - len(nb_auto)),
            'avg_duration_s':   round(float(nb_auto['duration_sec'].mean()), 1) if len(nb_auto) > 0 else round(float(nb_df['duration_sec'].mean()), 1),
            'avg_input_gb':     round(float(nb_df.get('input_gb', pd.Series([0])).mean()), 4),
            'avg_shuffle_gb':   round(float(nb_df.get('shuffle_gb', pd.Series([0])).mean()), 4),
            'is_orchestrator':  bool(nb_df['is_orchestrator'].any()) if 'is_orchestrator' in nb_df else False,
            'success_rate':     round(float(
                len(nb_df[nb_df['status'].isin(['Succeeded','Success'])]) / len(nb_df) * 100
            ), 0),
        }))
    all_nb_details[ws_name] = nb_details

    # ── Stats ────────────────────────────────────────────────
    avg_auto_dur = round(float(automated_df['duration_sec'].mean()), 1) if n_automated > 0 else None

    stats = {
        'has_notebooks':             True,
        'nb_count':                  int(df['notebook_name'].nunique()),
        'sample_size':               int(len(df)),
        'automated_sessions':        n_automated,
        'interactive_sessions':      n_interactive,
        'mostly_interactive':        mostly_interactive,
        'vcores_p95':                vcores_p95_final,
        'idle_pct_avg':              idle_pct_final,
        'vcores_source':             vcores_source,
        'avg_duration_sec':          round(float(df['duration_sec'].mean()), 1),
        'avg_duration_automated_sec':avg_auto_dur,
        'p95_duration_sec':          round(float(analysis_df['duration_sec'].quantile(0.95)), 1),
        'avg_input_gb':              round(float(df.get('input_gb',   pd.Series([0])).mean()), 4),
        'avg_output_gb':             round(float(df.get('output_gb',  pd.Series([0])).mean()), 4),
        'avg_shuffle_gb':            round(float(df.get('shuffle_gb', pd.Series([0])).mean()), 4),
        'max_parallel':              int(calc_max_parallel(df)),
        'is_orchestrator_ws':        is_orch_ws,
        'top_notebook':              str(df['notebook_name'].value_counts().index[0]),
        'top_user':                  str(df['submitted_by'].value_counts().index[0]),
        **pool,
    }

    rec = recommend_pool(stats)
    all_results.append(fix_types({'workspace_name': ws_name, 'workspace_id': ws_id, **stats, **rec}))

    src_icon = {'monitoring':'🔬','stages':'📊','none':'⚪'}.get(vcores_source,'⚪')
    dev_tag  = f' [{n_interactive} dev]' if n_interactive > 0 else ''
    orch_tag = ' [ORCH]' if is_orch_ws else ''
    print(f'{len(df)} sessions ({n_automated}auto+{n_interactive}dev) → {rec["action"]} ({rec["rec_node_size"]}x{rec["rec_max_nodes"]}) {rec["cu_saving_mo"]:+.0f} CU/mo {src_icon}{orch_tag}')


results_df = pd.DataFrame(all_results)
print(f'\n✅ Analysis complete — {len(results_df)} workspaces')
print(f'   🔬 Monitoring : {len([r for r in all_results if r.get("vcores_source")=="monitoring"])} workspaces')
print(f'   📊 Stages     : {len([r for r in all_results if r.get("vcores_source")=="stages"])} workspaces')
print(f'   ⚪ No data    : {len([r for r in all_results if r.get("vcores_source")=="none"])} workspaces')

In [ ]:
# ============================================================
# CELL 5: DASHBOARD
# ============================================================

def render_dashboard(df, nb_details):
    active_df = df[df['has_notebooks'] == True].copy()
    nodata_df = df[df['has_notebooks'] != True].copy()

    active_json = json.dumps(active_df.fillna(0).to_dict(orient='records'))
    nodata_json = json.dumps(nodata_df.fillna(0).to_dict(orient='records'))
    nb_json     = json.dumps(nb_details)

    total_ws     = len(df)
    active_ws    = len(active_df)
    down_n       = len(active_df[active_df['action'] == 'DOWNSIZE'])
    up_n         = len(active_df[active_df['action'] == 'UPSIZE'])
    ok_n         = len(active_df[active_df['action'] == 'OK'])
    total_saving = round(active_df[active_df['cu_saving_mo'] > 0]['cu_saving_mo'].sum(), 1)
    high_conf    = len(active_df[active_df['confidence'] == 'HIGH'])
    mon_count    = len(active_df[active_df.get('vcores_source','none') == 'monitoring']) if 'vcores_source' in active_df.columns else 0
    ts           = datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')

    CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{
  --bg:#f0f2f5;--card:#fff;--card2:#f8f9fb;--border:#e2e6ea;
  --text:#1a2332;--muted:#6b7a90;
  --green:#00875a;--gbg:#e3fcef;--gbd:#00875a33;
  --red:#de350b;--rbg:#ffebe6;--rbd:#de350b33;
  --amber:#ff8b00;--abg:#fff7e6;--abd:#ff8b0033;
  --blue:#0052cc;--bbg:#deebff;--bbd:#0052cc33;
  --purple:#5243aa;--pbg:#eae6ff;--pbd:#5243aa33;
  --grey:#f4f5f7;--grbd:#dfe1e6;
  --mono:'JetBrains Mono',monospace;--sans:'Inter',sans-serif;
  --r:8px;--sh:0 1px 3px rgba(0,0,0,.08);
}
body{background:var(--bg);font-family:var(--sans);color:var(--text);padding:20px;font-size:13px;line-height:1.5}
.tab-nav{display:flex;gap:0;border-bottom:2px solid var(--border);margin-bottom:20px}
.tab-btn{padding:10px 20px;font-size:13px;font-weight:600;color:var(--muted);background:transparent;border:none;cursor:pointer;border-bottom:3px solid transparent;margin-bottom:-2px;transition:all .15s}
.tab-btn:hover{color:var(--text)}
.tab-btn.active{color:var(--blue);border-bottom-color:var(--blue)}
.tab-pane{display:none}.tab-pane.active{display:block}
.hdr{display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:16px}
.hdr-title{font-size:18px;font-weight:700;display:flex;align-items:center;gap:8px}
.hdr-sub{font-size:11px;color:var(--muted);margin-top:3px;font-family:var(--mono)}
.ts{font-family:var(--mono);font-size:10px;color:var(--muted);background:var(--card);border:1px solid var(--border);padding:5px 10px;border-radius:var(--r);box-shadow:var(--sh)}
.cards{display:grid;grid-template-columns:repeat(5,1fr);gap:10px;margin-bottom:12px}
.card{background:var(--card);border:1px solid var(--border);border-radius:var(--r);padding:14px 16px;box-shadow:var(--sh);border-left:4px solid transparent}
.c-tot{border-left-color:var(--blue)}.c-dn{border-left-color:var(--green)}
.c-up{border-left-color:var(--red)}.c-ok{border-left-color:var(--muted)}.c-cu{border-left-color:var(--amber)}
.cv{font-size:26px;font-weight:700;font-family:var(--mono);line-height:1;margin-bottom:3px}
.c-tot .cv{color:var(--blue)}.c-dn .cv{color:var(--green)}
.c-up .cv{color:var(--red)}.c-ok .cv{color:var(--muted)}.c-cu .cv{color:var(--amber)}
.cl{font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:1px;color:var(--muted)}
.cs{font-size:11px;color:var(--muted);margin-top:1px}
.banner{background:var(--gbg);border:1px solid var(--gbd);border-radius:var(--r);padding:12px 18px;display:flex;align-items:center;gap:24px;margin-bottom:12px;flex-wrap:wrap}
.bv{font-family:var(--mono);font-size:22px;font-weight:700;color:var(--green)}
.bl{font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:1.5px;color:var(--green);margin-bottom:1px}
.bs{font-size:11px;color:var(--muted);margin-top:1px}
.bdiv{width:1px;height:36px;background:var(--border)}
.src-bar{background:var(--bbg);border:1px solid var(--bbd);border-radius:var(--r);padding:9px 14px;margin-bottom:12px;font-size:11px;color:var(--blue);display:flex;align-items:center;gap:16px;flex-wrap:wrap}
.fb-wrap{display:flex;align-items:center;gap:8px;margin-bottom:10px;flex-wrap:wrap}
.fbl{font-size:10px;font-weight:700;color:var(--muted);text-transform:uppercase;letter-spacing:1px}
.fb{font-size:11px;font-weight:600;padding:4px 12px;border-radius:20px;border:1.5px solid var(--border);background:var(--card);color:var(--muted);cursor:pointer;transition:all .15s}
.fb:hover{border-color:var(--blue);color:var(--blue)}
.fb.active{background:var(--blue);border-color:var(--blue);color:#fff}
.fb.fa-dn.active{background:var(--green);border-color:var(--green)}
.fb.fa-up.active{background:var(--red);border-color:var(--red)}
.fb.fa-ok.active{background:var(--muted);border-color:var(--muted)}
.si{font-size:11px;background:var(--card);border:1.5px solid var(--border);color:var(--text);padding:4px 12px;border-radius:20px;outline:none;width:200px;margin-left:auto}
.si:focus{border-color:var(--blue)}.si::placeholder{color:var(--muted)}
.tw{background:var(--card);border:1px solid var(--border);border-radius:var(--r);overflow:hidden;box-shadow:var(--sh);margin-bottom:10px}
.tw-hd{display:flex;justify-content:space-between;align-items:center;padding:11px 14px;border-bottom:1px solid var(--border);background:var(--card2)}
.tw-hd h2{font-size:12px;font-weight:700}.tc{font-family:var(--mono);font-size:10px;color:var(--muted)}
table{width:100%;border-collapse:collapse}
thead th{font-size:9px;font-weight:700;letter-spacing:1px;text-transform:uppercase;color:var(--muted);padding:8px 10px;text-align:left;border-bottom:1.5px solid var(--border);background:var(--card2);cursor:pointer;white-space:nowrap;user-select:none}
thead th:hover{color:var(--text)}
tbody tr{border-bottom:1px solid var(--border);cursor:pointer;transition:background .1s}
tbody tr:last-child{border-bottom:none}
tbody tr:hover{background:var(--bbg)}
tbody tr.is-orch{background:#faf8ff}
tbody tr.is-orch:hover{background:var(--pbg)}
tbody tr.has-dev{background:#fffdf0}
tbody td{padding:8px 10px;white-space:nowrap;vertical-align:middle}
.tn{font-weight:500;color:var(--blue);font-size:11px;max-width:170px;overflow:hidden;text-overflow:ellipsis}
.tm{font-family:var(--mono);font-size:10px;color:var(--muted)}
.ot{display:inline-block;font-size:8px;font-weight:700;background:var(--pbg);color:var(--purple);border:1px solid var(--pbd);padding:1px 4px;border-radius:3px;margin-left:3px;vertical-align:middle}
.dt-tag{display:inline-block;font-size:8px;font-weight:700;background:var(--abg);color:var(--amber);border:1px solid var(--abd);padding:1px 4px;border-radius:3px;margin-left:3px;vertical-align:middle}
.badge{display:inline-flex;align-items:center;gap:3px;font-size:9px;font-weight:700;letter-spacing:.5px;padding:2px 7px;border-radius:3px;border:1px solid}
.bDOWNSIZE{background:var(--gbg);color:var(--green);border-color:var(--gbd)}
.bUPSIZE{background:var(--rbg);color:var(--red);border-color:var(--rbd)}
.bOK{background:var(--grey);color:var(--muted);border-color:var(--grbd)}
.bNO_DATA{background:var(--abg);color:var(--amber);border-color:var(--abd)}
.pc{display:flex;align-items:center;gap:4px;font-family:var(--mono);font-size:9px}
.pc-c{color:var(--muted);text-decoration:line-through}.pc-a{color:var(--amber)}.pc-r{font-weight:700;color:var(--text)}
.cu-pos{font-family:var(--mono);font-size:10px;font-weight:600;color:var(--green)}
.cu-neg{font-family:var(--mono);font-size:10px;font-weight:600;color:var(--red)}
.cu-zer{font-family:var(--mono);font-size:10px;color:var(--muted)}
.conf-HIGH{font-family:var(--mono);font-size:9px;color:var(--green);font-weight:700}
.conf-MED{font-family:var(--mono);font-size:9px;color:var(--amber);font-weight:700}
.conf-LOW{font-family:var(--mono);font-size:9px;color:var(--muted)}
.nd-sec{margin-top:4px}
.nd-tog{display:flex;align-items:center;gap:8px;padding:9px 14px;background:var(--card);border:1px solid var(--border);border-radius:var(--r);cursor:pointer;font-size:12px;font-weight:500;color:var(--muted);box-shadow:var(--sh)}
.nd-tog:hover{color:var(--text)}.arr{transition:transform .2s;font-size:10px}
.nd-tog.open .arr{transform:rotate(90deg)}
.nd-body{display:none;background:var(--card);border:1px solid var(--border);border-top:none;border-radius:0 0 var(--r) var(--r);overflow:hidden}
.nd-body.vis{display:block}
.dp{background:var(--card);border:1px solid var(--border);border-radius:var(--r);padding:20px;display:none;margin-top:10px;box-shadow:var(--sh)}
.dp.vis{display:block}
.dh{display:flex;justify-content:space-between;align-items:center;margin-bottom:14px}
.dt{font-size:14px;font-weight:700;color:var(--blue);display:flex;align-items:center;gap:8px}
.dc{font-size:11px;cursor:pointer;border:1px solid var(--border);background:transparent;padding:3px 10px;border-radius:var(--r);color:var(--text)}
.dc:hover{border-color:var(--red);color:var(--red);background:var(--rbg)}
.dg{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:14px}
.dm{background:var(--card2);border:1px solid var(--border);border-radius:var(--r);padding:11px 13px}
.dmv{font-family:var(--mono);font-size:19px;font-weight:700;color:var(--text);line-height:1;margin-bottom:2px}
.dml{font-size:9px;font-weight:700;text-transform:uppercase;letter-spacing:1px;color:var(--muted)}
.dm-warn .dmv{color:var(--amber)}
.rec-box{border-radius:var(--r);padding:14px 16px;margin-bottom:12px}
.rec-dn{background:var(--gbg);border:1px solid var(--gbd)}
.rec-up{background:var(--rbg);border:1px solid var(--rbd)}
.rec-ok{background:var(--grey);border:1px solid var(--grbd)}
.rec-ttl{font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:8px}
.rec-dn .rec-ttl{color:var(--green)}.rec-up .rec-ttl{color:var(--red)}.rec-ok .rec-ttl{color:var(--muted)}
.rec-grid{display:flex;gap:18px;flex-wrap:wrap}
.ri{min-width:100px}.ril{font-size:9px;font-weight:700;text-transform:uppercase;letter-spacing:1px;color:var(--muted);margin-bottom:2px}
.riv{font-family:var(--mono);font-size:14px;font-weight:700}
.rec-dn .riv{color:var(--green)}.rec-up .riv{color:var(--red)}.rec-ok .riv{color:var(--muted)}
.exp-box{background:var(--card2);border:1px solid var(--border);border-radius:var(--r);padding:11px 14px;margin-bottom:12px}
.exp-box h4{font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:1px;color:var(--muted);margin-bottom:7px}
.exp-item{display:flex;gap:7px;margin-bottom:4px;font-size:12px}
.exp-b{color:var(--blue);font-weight:700;flex-shrink:0}
.exp-warn{color:var(--amber)}
.conf-box{background:var(--bbg);border:1px solid var(--bbd);border-radius:var(--r);padding:12px 16px;margin-bottom:12px}
.conf-box h4{font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:1px;color:var(--blue);margin-bottom:8px}
.step{display:flex;gap:9px;margin-bottom:5px;font-size:12px}
.sn{background:var(--blue);color:#fff;border-radius:50%;width:17px;height:17px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;flex-shrink:0;margin-top:1px}
.sk{font-family:var(--mono);font-weight:700;color:var(--blue)}
.nb-sec{margin-top:12px}
.nb-sec h4{font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:1px;color:var(--muted);margin-bottom:7px}
.nbt{width:100%;border-collapse:collapse;font-size:11px}
.nbt thead th{background:var(--card2);border:1px solid var(--border);padding:6px 9px;text-align:left;font-size:9px;font-weight:700;letter-spacing:1px;text-transform:uppercase;color:var(--muted)}
.nbt tbody td{border:1px solid var(--border);padding:6px 9px}
.nbt tbody tr:nth-child(odd){background:var(--card2)}
.nbt tbody tr:hover{background:var(--bbg)}
/* ABOUT PAGE */
.about-wrap{max-width:860px;margin:0 auto}
.about-section{background:var(--card);border:1px solid var(--border);border-radius:var(--r);padding:24px 28px;margin-bottom:16px;box-shadow:var(--sh)}
.about-section h2{font-size:16px;font-weight:700;color:var(--text);margin-bottom:12px;display:flex;align-items:center;gap:8px}
.about-section h3{font-size:13px;font-weight:700;color:var(--text);margin:14px 0 6px}
.about-section p{font-size:13px;color:var(--muted);margin-bottom:8px;line-height:1.6}
.about-section ul{padding-left:18px;font-size:13px;color:var(--muted);line-height:1.8}
.about-table{width:100%;border-collapse:collapse;font-size:12px;margin:10px 0}
.about-table thead th{background:var(--card2);border:1px solid var(--border);padding:8px 12px;text-align:left;font-size:10px;font-weight:700;letter-spacing:1px;text-transform:uppercase;color:var(--muted)}
.about-table tbody td{border:1px solid var(--border);padding:8px 12px;vertical-align:top}
.about-table tbody tr:nth-child(odd){background:var(--card2)}
.formula-box{background:var(--card2);border:1px solid var(--border);border-left:4px solid var(--blue);border-radius:var(--r);padding:12px 16px;margin:10px 0;font-family:var(--mono);font-size:12px;color:var(--text);line-height:1.8}
.formula-box .comment{color:var(--muted);font-style:italic}
.pill{display:inline-block;font-size:10px;font-weight:700;padding:2px 8px;border-radius:12px;margin:2px}
.pill-green{background:var(--gbg);color:var(--green);border:1px solid var(--gbd)}
.pill-amber{background:var(--abg);color:var(--amber);border:1px solid var(--abd)}
.pill-blue{background:var(--bbg);color:var(--blue);border:1px solid var(--bbd)}
.pill-muted{background:var(--grey);color:var(--muted);border:1px solid var(--grbd)}
.limit-note{background:var(--rbg);border:1px solid var(--rbd);border-radius:var(--r);padding:10px 14px;margin:10px 0;font-size:12px;color:var(--red)}
.limit-note strong{font-weight:700}
.tip-note{background:var(--gbg);border:1px solid var(--gbd);border-radius:var(--r);padding:10px 14px;margin:10px 0;font-size:12px;color:var(--green)}
.empty{text-align:center;padding:28px;color:var(--muted);font-size:12px}
"""

    html = f"""<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8"><style>{CSS}</style></head><body>

<div class="tab-nav">
  <button class="tab-btn active" onclick="showTab('dashboard',this)">📊 Dashboard</button>
  <button class="tab-btn" onclick="showTab('about',this)">ℹ About & Methodology</button>
</div>

<!-- ═══════════════════════════════════════════════════════ DASHBOARD TAB -->
<div class="tab-pane active" id="tab-dashboard">

<div class="hdr">
  <div>
    <div class="hdr-title">⚡ Fabric Spark Pool Optimiser</div>
    <div class="hdr-sub">{total_ws} workspaces · {active_ws} with sessions · Last {DAYS_BACK} days · {ts}</div>
  </div>
  <span class="ts">⟳ {ts}</span>
</div>

<div class="cards">
  <div class="card c-tot"><div class="cv">{active_ws}</div><div class="cl">Active</div><div class="cs">with Spark sessions</div></div>
  <div class="card c-dn"><div class="cv">{down_n}</div><div class="cl">Downsize</div><div class="cs">oversized pools</div></div>
  <div class="card c-up"><div class="cv">{up_n}</div><div class="cl">Upsize</div><div class="cs">undersized pools</div></div>
  <div class="card c-ok"><div class="cv">{ok_n}</div><div class="cl">OK</div><div class="cs">correctly sized</div></div>
  <div class="card c-cu"><div class="cv">{total_saving:,.0f}</div><div class="cl">CU saving</div><div class="cs">estimated / month</div></div>
</div>

<div class="banner">
  <div><div class="bl">💰 Total monthly CU saving</div><div class="bv">{total_saving:,.1f} CU</div><div class="bs">if all DOWNSIZE recommendations applied · based on last {DAYS_BACK} days</div></div>
  <div class="bdiv"></div>
  <div><div class="bl">Changes needed</div><div class="bv">{down_n + up_n}</div><div class="bs">{down_n} downsize · {up_n} upsize</div></div>
  <div class="bdiv"></div>
  <div><div class="bl">High confidence</div><div class="bv">{high_conf}</div><div class="bs">≥ {MIN_RUNS * 4} sessions analysed</div></div>
</div>

<div class="src-bar">
  <strong>Data sources:</strong>
  <span>🔬 <strong>{mon_count}</strong> workspaces — Workspace Monitoring (real vCores)</span>
  <span>📊 Stage data (GB read/written/shuffle)</span>
  <span>⚪ Duration + parallelism only</span>
  <span style="color:var(--muted);font-size:10px">| ⚠ Interactive dev sessions excluded from timing/CU analysis</span>
</div>

<div class="fb-wrap">
  <span class="fbl">Show:</span>
  <button class="fb active" onclick="ft('ALL',this)">All ({active_ws})</button>
  <button class="fb fa-dn" onclick="ft('DOWNSIZE',this)">↓ Downsize ({down_n})</button>
  <button class="fb fa-up" onclick="ft('UPSIZE',this)">↑ Upsize ({up_n})</button>
  <button class="fb fa-ok" onclick="ft('OK',this)">✓ OK ({ok_n})</button>
  <input class="si" type="text" placeholder="Search workspace..." oninput="st(this.value)">
</div>

<div class="tw">
  <div class="tw-hd"><h2>Workspace Analysis</h2><span class="tc" id="tc">{active_ws} workspaces</span></div>
  <table><thead><tr>
    <th onclick="so('workspace_name')">WORKSPACE</th>
    <th onclick="so('action')">ACTION</th>
    <th>CURRENT</th><th>RECOMMENDED</th>
    <th onclick="so('avg_input_gb')">DATA IN</th>
    <th onclick="so('avg_shuffle_gb')">SHUFFLE</th>
    <th onclick="so('avg_duration_automated_sec')">AUTO DUR</th>
    <th onclick="so('interactive_sessions')">DEV SESSIONS</th>
    <th onclick="so('max_parallel')">MAX PAR</th>
    <th onclick="so('cu_saving_mo')">CU SAVING/MO</th>
    <th onclick="so('cu_current_mo')">CURRENT CU/MO</th>
    <th onclick="so('sample_size')">RUNS</th>
    <th onclick="so('confidence')">CONF</th>
    <th>SRC</th>
  </tr></thead><tbody id="tb"></tbody></table>
</div>

<div class="nd-sec" id="nd-sec" style="display:none">
  <div class="nd-tog" id="nd-tog" onclick="tnd()">
    <span class="arr">▶</span>
    <span id="nd-lbl">Workspaces without Spark sessions (0)</span>
    <span style="margin-left:auto;font-size:11px">Click to expand</span>
  </div>
  <div class="nd-body" id="nd-body">
    <table><thead><tr><th>WORKSPACE</th><th>CURRENT POOL</th><th>NOTEBOOKS</th><th>STATUS</th></tr></thead>
    <tbody id="nd-tb"></tbody></table>
  </div>
</div>

<div class="dp" id="dp">
  <div class="dh">
    <div class="dt" id="dt">—</div>
    <button class="dc" onclick="cd()">✕ Close</button>
  </div>
  <div class="dg" id="dg"></div>
  <div id="drec"></div>
  <div id="dexp"></div>
  <div id="dconf"></div>
  <div class="nb-sec" id="dnb"></div>
</div>

</div><!-- end dashboard tab -->

<!-- ═══════════════════════════════════════════════════════ ABOUT TAB -->
<div class="tab-pane" id="tab-about">
<div class="about-wrap">

  <div class="about-section">
    <h2>⚡ What is the Fabric Spark Pool Optimiser?</h2>
    <p>A Fabric Spark Notebook that analyses real Spark session data across your workspaces and recommends the optimal pool configuration — so you stop paying for compute you don't use, and stop throttling workloads that need more.</p>
    <p>It runs entirely inside Fabric, requires no external dependencies, and adapts to whatever data is available in your environment.</p>
    <div class="tip-note">✅ <strong>No lakehouse needed.</strong> No configuration needed. Just run all cells and scroll to the dashboard.</div>
  </div>

  <div class="about-section">
    <h2>📡 Data sources</h2>
    <table class="about-table">
      <thead><tr><th>Data</th><th>Source</th><th>Available</th><th>Notes</th></tr></thead>
      <tbody>
        <tr><td>Session duration</td><td>Livy Sessions API</td><td>✅ Always</td><td>Real wall-clock time</td></tr>
        <tr><td>GB read / written</td><td>Spark History /stages</td><td>✅ Always</td><td>inputBytes + outputBytes from all stages</td></tr>
        <tr><td>GB shuffled</td><td>Spark History /stages</td><td>✅ Always</td><td>shuffleReadBytes — indicates memory pressure</td></tr>
        <tr><td>Max concurrent sessions</td><td>Livy Sessions API</td><td>✅ Always</td><td>Used to size maxNodes</td></tr>
        <tr><td>vCores really used</td><td>Workspace Monitoring KQL</td><td>✅ If enabled</td><td>Most accurate — enables Tier 1 sizing</td></tr>
        <tr><td>vCores really used</td><td>resourceusage API</td><td>❌ Not available</td><td>Returns 500 for pipeline-triggered sessions (Fabric limitation)</td></tr>
        <tr><td>Sub-notebook breakdown</td><td>itemsnapshots / jobGroups</td><td>❌ Not available</td><td>All jobs appear under jobGroup=None when using runMultiple()</td></tr>
      </tbody>
    </table>
  </div>

  <div class="about-section">
    <h2>🔍 Session classification: Automated vs Interactive</h2>
    <p>One of the key problems in pool sizing is that <strong>developer sessions distort the data</strong>. A data engineer who leaves a notebook open for 2 hours while iterating on code creates a session with 7,200s duration — which would make the pool appear much more expensive than it actually is for production workloads.</p>
    <p>The optimiser classifies every session:</p>
    <table class="about-table">
      <thead><tr><th>Type</th><th>Criteria</th><th>Used for</th></tr></thead>
      <tbody>
        <tr>
          <td><span class="pill pill-blue">automated</span></td>
          <td>Service Principal submitter OR orchestrator name pattern OR duration ≤ {INTERACTIVE_THRESHOLD}s</td>
          <td>Duration, CU calculation, parallelism analysis</td>
        </tr>
        <tr>
          <td><span class="pill pill-amber">interactive dev</span></td>
          <td>User submitter AND duration > {INTERACTIVE_THRESHOLD}s</td>
          <td>GB data (still real) — excluded from timing/CU</td>
        </tr>
      </tbody>
    </table>
    <p>If <strong>>50% of sessions are interactive</strong>, the workspace is flagged and GB-based sizing (Tier 2) is forced regardless of vCores availability, using only automated session durations for the CU formula.</p>
    <div class="tip-note">💡 You can adjust <code>INTERACTIVE_THRESHOLD</code> in Cell 1 (default: {INTERACTIVE_THRESHOLD}s = 30 minutes).</div>
  </div>

  <div class="about-section">
    <h2>📊 Three-tier recommendation logic</h2>
    <p>The recommendation engine uses the best available data, falling back gracefully:</p>
    <table class="about-table">
      <thead><tr><th>Tier</th><th>Data used</th><th>nodeSize logic</th><th>Confidence</th></tr></thead>
      <tbody>
        <tr>
          <td><span class="pill pill-green">Tier 1</span><br>Workspace Monitoring</td>
          <td>Real vCores p95 from KQL</td>
          <td>Smallest node where vCores ≥ p95_real × 1.20</td>
          <td><span class="conf-HIGH">HIGH</span></td>
        </tr>
        <tr>
          <td><span class="pill pill-amber">Tier 2</span><br>Stage data</td>
          <td>GB/min throughput from stages</td>
          <td>&gt;5 GB/min → Large · &gt;1 GB/min → Medium · else → Small</td>
          <td><span class="conf-MED">MED</span></td>
        </tr>
        <tr>
          <td><span class="pill pill-muted">Tier 3</span><br>Duration only</td>
          <td>Session duration + parallelism</td>
          <td>Keep current nodeSize (cannot determine over/under)</td>
          <td><span class="conf-LOW">LOW</span></td>
        </tr>
      </tbody>
    </table>
    <p>Additionally, if <strong>shuffle GB > 1.5× input GB</strong>, nodeSize is bumped one level up to handle memory pressure from data redistribution.</p>
  </div>

  <div class="about-section">
    <h2>💰 CU saving calculation</h2>
    <p>Based on the official Microsoft formula: <strong>1 CU = 2 Spark vCores</strong>. Billing applies only during active session runtime.</p>
    <div class="formula-box">
      CU_per_session = cu_per_node_hr × max_nodes × avg_duration_hr<br>
      CU_per_month   = CU_per_session × runs_per_month<br><br>
      <span class="comment">// runs_per_month = sessions_in_period / (DAYS_BACK / 30)</span><br>
      <span class="comment">// avg_duration_hr = automated sessions only (interactive excluded)</span><br><br>
      saving = CU_current_month − CU_recommended_month<br>
      <span class="comment">// positive = saving (downsize)  ·  negative = extra cost (upsize)</span>
    </div>
    <p><strong>Why CU saving can be positive even on DOWNSIZE with more nodes:</strong> If nodeSize goes from Medium → Small (cheaper per node) but maxNodes goes up (more parallelism needed), the net effect depends on which change dominates. The dashboard shows both the action direction and the net CU impact.</p>
  </div>

  <div class="about-section">
    <h2>🏭 Orchestrator workspaces</h2>
    <p>In Fabric, notebooks triggered via <code>runMultiple()</code> or Data Factory pipelines run <strong>in the orchestrator workspace</strong>, not in the target workspace (Gold, Silver, etc.).</p>
    <p>This means:</p>
    <ul>
      <li>Gold/Silver workspaces show 0-2 sessions (manual runs only) → appear as NO_DATA</li>
      <li>The Control workspace shows all production sessions → the pool that matters</li>
      <li>Sub-notebook GB cannot be attributed individually (all jobs appear under jobGroup=None)</li>
    </ul>
    <p>A workspace is automatically flagged as an orchestrator if ≥30% of sessions match orchestrator name patterns (<code>{', '.join(ORCHESTRATOR_PATTERNS)}</code>) or ≥50% are submitted by a Service Principal.</p>
    <div class="limit-note"><strong>API limitation:</strong> The resourceusage endpoint returns HTTP 500 for pipeline-triggered sessions. vCores are only available via Workspace Monitoring KQL. To enable: Admin Portal → Workspace Settings → Monitoring.</div>
  </div>

  <div class="about-section">
    <h2>⚙️ Node size reference</h2>
    <table class="about-table">
      <thead><tr><th>Node Size</th><th>vCores</th><th>RAM</th><th>CU/hr per node</th><th>Best for</th></tr></thead>
      <tbody>
        <tr><td><strong>Small</strong></td><td>4</td><td>32 GB</td><td>2</td><td>Small dimension tables, reference data, dev</td></tr>
        <tr style="background:var(--bbg)"><td><strong>Medium ★</strong></td><td>8</td><td>64 GB</td><td>4</td><td>Default · Most production workloads · Orchestrators</td></tr>
        <tr><td><strong>Large</strong></td><td>16</td><td>128 GB</td><td>8</td><td>Large fact tables, heavy transformations</td></tr>
        <tr><td><strong>XLarge</strong></td><td>32</td><td>256 GB</td><td>16</td><td>Very large datasets, complex joins</td></tr>
        <tr><td><strong>XXLarge</strong></td><td>64</td><td>512 GB</td><td>32</td><td>Massive scale, ML workloads</td></tr>
      </tbody>
    </table>
    <p style="font-size:11px;color:var(--muted)">★ Default Starter Pool. Source: Microsoft Fabric official documentation (1 CU = 2 Spark vCores).</p>
  </div>

</div>
</div><!-- end about tab -->

<script>
const ACT={json.dumps(json.loads(active_json))};
const NOD={json.dumps(json.loads(nodata_json))};
const NBD={json.dumps(nb_details)};
let DATA=[...ACT],cf='ALL',cs='';
const AL={{DOWNSIZE:'↓ DOWNSIZE',UPSIZE:'↑ UPSIZE',OK:'✓ OK',NO_DATA:'? NO DATA'}};
const SI={{monitoring:'🔬',stages:'📊',none:'⚪'}};

function showTab(id,btn){{
  document.querySelectorAll('.tab-pane').forEach(p=>p.classList.remove('active'));
  document.querySelectorAll('.tab-btn').forEach(b=>b.classList.remove('active'));
  document.getElementById('tab-'+id).classList.add('active');
  btn.classList.add('active');
}}

function pcell(r){{
  if(r.action==='OK'||r.action==='NO_DATA')
    return `<span class="tm">${{r.cur_node_size}}×${{r.cur_max_nodes}}</span>`;
  return `<span class="pc-c" style="font-family:var(--mono);font-size:10px">${{r.cur_node_size}}×${{r.cur_max_nodes}}</span>`;
}}
function rcell(r){{
  if(r.action==='OK') return `<span class="tm">No change</span>`;
  if(r.action==='NO_DATA') return '—';
  return `<span class="pc"><span class="pc-a">→</span><span class="pc-r">${{r.rec_node_size}}×${{r.rec_max_nodes}}</span></span>`;
}}
function cusave(v, action){{
  if(!v&&v!==0) return '<span class="cu-zer">—</span>';
  if(action==='UPSIZE'){{
    return `<span class="cu-neg">+${{Math.abs(v).toFixed(0)}} CU/mo extra</span>`;
  }}
  if(v>0)  return `<span class="cu-pos">-${{v.toFixed(0)}} CU/mo</span>`;
  if(v<0)  return `<span class="cu-neg">+${{Math.abs(v).toFixed(0)}} CU/mo extra</span>`;
  return '<span class="cu-zer">0</span>';
}}

function rt(){{
  const fd=DATA.filter(r=>{{
    if(cf!=='ALL'&&r.action!==cf) return false;
    if(cs&&!r.workspace_name.toLowerCase().includes(cs.toLowerCase())) return false;
    return true;
  }});
  document.getElementById('tc').textContent=fd.length+' workspaces';
  const tb=document.getElementById('tb');
  if(!fd.length){{tb.innerHTML='<tr><td colspan="14"><div class="empty">No workspaces match.</div></td></tr>';return;}}
  tb.innerHTML=fd.map(r=>`
    <tr class="${{[r.is_orchestrator_ws?'is-orch':'',r.interactive_sessions>0?'has-dev':''].filter(Boolean).join(' ')}}"
        onclick="sd(${{JSON.stringify(r).replace(/"/g,'&quot;')}})">
      <td class="tn">${{r.workspace_name}}
        ${{r.is_orchestrator_ws?'<span class="ot">ORCH</span>':''}}
        ${{r.interactive_sessions>0?'<span class="dt-tag">'+r.interactive_sessions+' dev</span>':''}}
      </td>
      <td><span class="badge b${{r.action}}">${{AL[r.action]||r.action}}</span></td>
      <td>${{pcell(r)}}</td>
      <td>${{rcell(r)}}</td>
      <td class="tm">${{(r.avg_input_gb||0).toFixed(3)}} GB</td>
      <td class="tm">${{(r.avg_shuffle_gb||0).toFixed(3)}} GB</td>
      <td class="tm">${{r.avg_duration_automated_sec?r.avg_duration_automated_sec.toFixed(0)+'s (auto)':'—'}}</td>
      <td class="tm">${{r.interactive_sessions||0}}</td>
      <td class="tm">${{r.max_parallel||1}}</td>
      <td>${{cusave(r.cu_saving_mo, r.action)}}</td>
      <td class="tm">${{(r.cu_current_mo||0).toFixed(0)}} CU</td>
      <td class="tm">${{r.sample_size||0}}</td>
      <td class="conf-${{r.confidence||'LOW'}}">${{r.confidence||'LOW'}}</td>
      <td style="font-size:11px">${{SI[r.vcores_source||'none']||'⚪'}} ${{r.vcores_source||'none'}}</td>
    </tr>`).join('');
}}

function rnd(){{
  if(!NOD.length) return;
  document.getElementById('nd-sec').style.display='block';
  document.getElementById('nd-lbl').textContent='Workspaces without Spark sessions ('+NOD.length+')';
  document.getElementById('nd-tb').innerHTML=NOD.map(r=>`
    <tr>
      <td class="tn">${{r.workspace_name}}</td>
      <td class="tm">${{r.cur_node_size||'?'}}×${{r.cur_max_nodes||'?'}}</td>
      <td class="tm">${{r.nb_count||0}} notebooks</td>
      <td><span class="badge bNO_DATA">? NO SESSIONS</span></td>
    </tr>`).join('');
}}

function tnd(){{
  const t=document.getElementById('nd-tog'),b=document.getElementById('nd-body');
  t.classList.toggle('open');b.classList.toggle('vis');
}}

function ft(a,btn){{cf=a;document.querySelectorAll('.fb').forEach(b=>b.classList.remove('active'));btn.classList.add('active');rt();}}
function st(v){{cs=v;rt();}}

let sk='cu_saving_mo',sa=false;
function so(k){{
  sa=sk===k?!sa:false;sk=k;
  DATA.sort((a,b)=>{{const av=a[k]??0,bv=b[k]??0;const c=typeof av==='string'?av.localeCompare(bv):av-bv;return sa?c:-c;}});
  rt();
}}

function sd(r){{
  if(typeof r==='string') r=JSON.parse(r.replace(/&quot;/g,'"'));
  document.getElementById('dt').innerHTML='⚡ '+r.workspace_name+
    (r.is_orchestrator_ws?' <span class="ot">ORCHESTRATOR</span>':'')+
    (r.mostly_interactive?' <span class="dt-tag">mostly interactive</span>':'');

  const autoNote=r.avg_duration_automated_sec
    ?r.avg_duration_automated_sec.toFixed(0)+'s (automated only)'
    :(r.avg_duration_sec||0).toFixed(0)+'s (all)';
  const devWarn=r.mostly_interactive?'dm-warn':'';

  document.getElementById('dg').innerHTML=`
    <div class="dm"><div class="dmv">${{(r.vcores_p95||0)>0?r.vcores_p95.toFixed(1):'N/A'}}</div><div class="dml">vCores p95 ${{SI[r.vcores_source||'none']}}</div></div>
    <div class="dm"><div class="dmv">${{r.idle_pct_avg!=null?((r.idle_pct_avg)*100).toFixed(0)+'%':'N/A'}}</div><div class="dml">Idle cores avg</div></div>
    <div class="dm"><div class="dmv">${{(r.avg_input_gb||0).toFixed(3)}} GB</div><div class="dml">Avg data read</div></div>
    <div class="dm"><div class="dmv">${{(r.avg_shuffle_gb||0).toFixed(3)}} GB</div><div class="dml">Avg shuffle</div></div>
    <div class="dm ${{devWarn}}"><div class="dmv">${{autoNote}}</div><div class="dml">Avg duration${{r.mostly_interactive?' ⚠':''}}</div></div>
    <div class="dm"><div class="dmv">${{r.automated_sessions||0}} / ${{r.interactive_sessions||0}}</div><div class="dml">Auto / Interactive sessions</div></div>
    <div class="dm"><div class="dmv">${{r.max_parallel||1}}</div><div class="dml">Max parallel sessions</div></div>
    <div class="dm"><div class="dmv">${{(r.cu_current_mo||0).toFixed(0)}}</div><div class="dml">Current CU / month</div></div>`;

  const rc=r.action==='DOWNSIZE'?'rec-dn':r.action==='UPSIZE'?'rec-up':'rec-ok';
  const sav=r.action==='UPSIZE'
    ?`<div class="ri"><div class="ril">Extra cost (upsize)</div><div class="riv" style="color:var(--red)">+${{Math.abs(r.cu_saving_mo||0).toFixed(0)}} CU/mo</div></div>`
    :(r.cu_saving_mo||0)>0
    ?`<div class="ri"><div class="ril">Monthly saving</div><div class="riv">${{(r.cu_saving_mo).toFixed(0)}} CU/mo</div></div>`
    :'';
  document.getElementById('drec').innerHTML=`
    <div class="rec-box ${{rc}}">
      <div class="rec-ttl">${{r.action==='DOWNSIZE'?'↓ DOWNSIZE RECOMMENDED':r.action==='UPSIZE'?'↑ UPSIZE RECOMMENDED':'✓ POOL SIZE IS CORRECT'}}</div>
      <div class="rec-grid">
        <div class="ri"><div class="ril">Current pool</div><div class="riv" style="color:var(--muted)">${{r.cur_node_size}} × ${{r.cur_max_nodes}}</div></div>
        <div class="ri"><div class="ril">Recommended</div><div class="riv">${{r.rec_node_size||r.cur_node_size}} × ${{r.rec_max_nodes||r.cur_max_nodes}}</div></div>
        <div class="ri"><div class="ril">Min nodes</div><div class="riv">1</div></div>
        ${{sav}}
        <div class="ri"><div class="ril">Confidence</div><div class="riv" style="color:${{r.confidence==='HIGH'?'var(--green)':r.confidence==='MED'?'var(--amber)':'var(--muted)'}}">${{r.confidence}}</div></div>
      </div>
    </div>`;

  const parts=(r.explanation||'').split(' | ').filter(Boolean);
  document.getElementById('dexp').innerHTML=parts.length?`
    <div class="exp-box">
      <h4>📊 Why this recommendation</h4>
      ${{parts.map(p=>`<div class="exp-item"><span class="exp-b ${{p.startsWith('⚠')||p.includes('interactive')?'exp-warn':''}}">→</span><span>${{p}}</span></div>`).join('')}}
    </div>`:'';

  const steps=r.config_steps||[];
  document.getElementById('dconf').innerHTML=steps.length?`
    <div class="conf-box">
      <h4>🔧 How to apply this configuration</h4>
      ${{steps.map((s,i)=>{{
        const t=s.replace(/(Small|Medium|Large|XLarge|XXLarge)/g,'<span class="sk">$1</span>')
                 .replace(/([0-9]+ nodes?|[0-9]+ vCores?|[0-9]+ GB|Yes)/g,'<span class="sk">$1</span>');
        return `<div class="step"><div class="sn">${{i+1}}</div><div>${{t}}</div></div>`;
      }}).join('')}}
    </div>`:'';

  const nbs=NBD[r.workspace_name]||[];
  document.getElementById('dnb').innerHTML=nbs.length?`
    <h4>📓 Notebook breakdown (${{nbs.length}})</h4>
    <table class="nbt"><thead><tr>
      <th>NOTEBOOK</th><th>RUNS (auto/dev)</th><th>AVG DUR (auto)</th>
      <th>DATA IN</th><th>SHUFFLE</th><th>TYPE</th><th>SUCCESS</th>
    </tr></thead><tbody>
    ${{nbs.map(n=>`<tr>
      <td style="font-weight:500">${{n.notebook_name}}</td>
      <td class="tm">${{n.automated_runs}}/${{n.interactive_runs}}</td>
      <td class="tm">${{n.avg_duration_s}}s</td>
      <td class="tm">${{n.avg_input_gb}} GB</td>
      <td class="tm">${{n.avg_shuffle_gb}} GB</td>
      <td>${{n.is_orchestrator?'<span class="ot">ORCH</span>':'notebook'}}</td>
      <td class="tm" style="color:${{n.success_rate>=90?'var(--green)':n.success_rate>=70?'var(--amber)':'var(--red)'}}">${{n.success_rate}}%</td>
    </tr>`).join('')}}
    </tbody></table>
    ${{r.is_orchestrator_ws?'<p style="font-size:11px;color:var(--muted);margin-top:6px">ℹ Orchestrator: sub-notebooks share the Spark session via runMultiple(). Individual sub-notebook GB attribution not available via Fabric API.</p>':''}}`:
    '<div class="empty">No notebook detail available</div>';

  const p=document.getElementById('dp');p.classList.add('vis');p.scrollIntoView({{behavior:'smooth',block:'start'}});
}}

function cd(){{document.getElementById('dp').classList.remove('vis');}}
so('cu_saving_mo');
rnd();
</script></body></html>"""
    return html


display(HTML(render_dashboard(results_df, all_nb_details)))


